## 00. Imports & Config

In [15]:
import os
import sys
import importlib
import pandas as pd 
import warnings

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from api_retrieval.markets.exus.database_extractor import DatabaseExtractor

warnings.filterwarnings('ignore')

db = DatabaseExtractor(
    server="exusbigcloud.database.windows.net",
    username="exusbigcloud_readonly",
    password="H8blZuJ0tju2"
)

In [ ]:
#hey

In [ ]:
quarter = '26Q2'

countries = ['DE', 'IT', 'ES', 'FR', 'PT', 'PL', 'GB'] #Ireland is missing, so follow up with Energy Management in case they add it at any point


In [23]:
# Output path for quarterly prices

path = f"../../../outputs/quarterly/exus_monthly_prices_{quarter}.xlsx"

In [17]:
from common_libs.utils.utils_dates import get_start_date_from_quarterly

start_date, end_date = get_start_date_from_quarterly(quarter)

start_date, end_date

('2024-07-01', '2026-06-30')

## 01. Extract DAM prices data

In [ ]:
df_DAM = db.query_table(
    "bigcloud_db",
    "dbo.EuropeDAMActuals",
    order_by="UTCTime DESC",
    start=start_date,
    end=end_date,
    filters={"CountryCode": countries}
)

In [20]:
df_DAM_mod = df_DAM.copy()

# To standarize time for quarterhourly data and hourly data, we will group data by hour and take the mean of the quarterhourly data. This way, we will have a single value for each hour, which will be comparable to the hourly data.

# Parse LocalTime as UTC-aware datetime and drop the tz component before bucketing
df_DAM_mod["LocalTime"] = pd.to_datetime(df_DAM_mod["LocalTime"],
    format='ISO8601',
    utc=True
)

df_DAM_mod['LocalDate'] = df_DAM_mod['LocalTime'].dt.date
df_DAM_mod["LocalHour"] = df_DAM_mod["LocalTime"].dt.hour

df_DAM_mod['Month'] = df_DAM_mod['LocalTime'].dt.to_period('M')
df_DAM_mod['Year'] = df_DAM_mod['LocalTime'].dt.year
 
# Average to hourly
df_hour = (
    df_DAM_mod
    .groupby(["CountryCode", "PriceAreaCode", "Unit", "Market", "LocalDate", "LocalHour"], as_index=False)["Price"]
    .mean()
)

In [21]:
# Now get the average monthly prices for each country and price area. This will be useful for the analysis of the price trends over time.

df_hour['LocalDate'] = pd.to_datetime(df_hour['LocalDate'])
df_hour['Month'] = df_hour['LocalDate'].dt.month
df_hour['Year'] = df_hour['LocalDate'].dt.year

df_monthly = (
    df_hour
    .groupby(["CountryCode", "PriceAreaCode", "Unit", "Market", "Year", "Month"], as_index=False)["Price"]
    .mean()
)

In [ ]:

df_pivot = df_monthly.pivot_table(columns=['Year', 'Month'], index=['CountryCode', 'PriceAreaCode', 'Market', 'Unit'], values='Price')

df_pivot.to_excel(path)

df_pivot

Year                                                   2024              \
Month                                                    7           8    
CountryCode PriceAreaCode   Market      Unit                              
DE          DE-LU           DAM         €/MWh     67.710000   82.034933   
ES          ES              DAM         €/MWh     72.433118   91.098710   
FR          FR              DAM         €/MWh     47.468591   54.634522   
GB          GB              DAM         GBP/MWh   69.788333   59.480108   
IT          IT-Calabria     DAM         €/MWh    118.188181  131.478357   
            IT-Centre-North DAM         €/MWh    117.777850  131.074170   
            IT-Centre-South DAM         €/MWh    118.156738  131.252706   
            IT-North        DAM         €/MWh    107.816742  124.625286   
            IT-SACOAC       DAM         €/MWh    117.614887  131.058340   
            IT-SACODC       DAM         €/MWh    118.156738  131.179958   
            IT-Sardinia     DAM         €/MWh    117.614887  131.058340   
            IT-Sicily       DAM         €/MWh    118.931957  131.456757   
            IT-South        DAM         €/MWh    118.156738  131.252706   
PL          PL              DAM         €/MWh    109.005161  100.116196   
                            PL FIXING I PLN/MWh  478.670820  426.740914   
PT          PT              DAM         €/MWh     76.425638   94.736613   

Year                                                                     \
Month                                                    9           10   
CountryCode PriceAreaCode   Market      Unit                              
DE          DE-LU           DAM         €/MWh     78.064889   86.292997   
ES          ES              DAM         €/MWh     72.537903   68.491129   
FR          FR              DAM         €/MWh     52.709716   62.984631   
GB          GB              DAM         GBP/MWh   75.580292   83.613266   
IT          IT-Calabria     DAM         €/MWh    118.534615  119.248088   
            IT-Centre-North DAM         €/MWh    117.993001  115.937337   
            IT-Centre-South DAM         €/MWh    117.400578  118.514381   
            IT-North        DAM         €/MWh    115.180749  114.835569   
            IT-SACOAC       DAM         €/MWh    114.177440  117.703653   
            IT-SACODC       DAM         €/MWh    199.741166  214.990222   
            IT-Sardinia     DAM         €/MWh    114.177440  117.703653   
            IT-Sicily       DAM         €/MWh    119.068804  134.290347   
            IT-South        DAM         €/MWh    117.164920  118.514381   
PL          PL              DAM         €/MWh     94.993222  103.157204   
                            PL FIXING I PLN/MWh  402.766250  450.398883   
PT          PT              DAM         €/MWh     78.600230   71.119433   

Year                                                                     \
Month                                                    11          12   
CountryCode PriceAreaCode   Market      Unit                              
DE          DE-LU           DAM         €/MWh    113.956889  108.184543   
ES          ES              DAM         €/MWh    104.426008  111.253360   
FR          FR              DAM         €/MWh    101.036597   97.961725   
GB          GB              DAM         GBP/MWh   97.072486   89.412245   
IT          IT-Calabria     DAM         €/MWh    130.936866  131.088658   
            IT-Centre-North DAM         €/MWh    130.937060  135.602476   
            IT-Centre-South DAM         €/MWh    130.670014  135.510490   
            IT-North        DAM         €/MWh    130.422496  135.267948   
            IT-SACOAC       DAM         €/MWh    130.670014  134.766308   
            IT-SACODC       DAM         €/MWh    133.484544  135.608952   
            IT-Sardinia     DAM         €/MWh    130.670014  134.766308   
            IT-Sicily       DAM         €/MWh    131.469970  131.121080   
            IT-South   